<div style="background:linear-gradient(120deg,#07062e 0%,#120c5c 60%,#2a1a9e 100%);border-radius:14px;padding:26px 30px;color:#ffffff;font-family:'Segoe UI',Arial,sans-serif;"><div style="font-size:12px;letter-spacing:3px;color:#4de1ff;font-weight:600;">MICROSOFT IGNITE 2026 &nbsp;&middot;&nbsp; LAB663</div><div style="font-size:30px;font-weight:700;margin:8px 0 4px 0;color:#ffffff;">Module 3 &middot; Multi-model vs multi-modal</div><div style="font-size:15px;color:#cfd4ff;">Set the stage &middot; about 8 minutes</div><div style="margin-top:16px;"><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">1</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">2</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#4de1ff;color:#07062e;">3</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">4</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">5</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">6</span></div><div style="height:6px;border-radius:3px;margin-top:18px;background:linear-gradient(90deg,#3cf26b,#4de1ff,#6c4dff,#ff4fd8,#ff9f1c,#ffd23f);"></div></div>

# 3. Multi-model vs multi-modal

**Time:** about 8 minutes. This is the last "set the stage" notebook.

**The question we're starting with:** what's the difference between multi-model and multi-modal, and when do I need each?

Real customers don't send tidy text questions. They send a photo of the tent in their garage and ask "does this one hold up in rain?" That one request needs **a model that can see** and **an agent that knows our manuals**. No single model call does both.

So here we try two ideas that show up everywhere in real systems. They sound alike and mean different things:

| | Multi-modal | Multi-model |
|---|---|---|
| The idea | The input (or output) is more than text | Several models work on one job |
| Example | A customer sends a photo | A small model looks, a big model answers |
| It answers | "What can I send in?" | "Which model handles which step?" |

And we'll finish by building a *tiny router by hand*, so we feel why a managed one is worth having.


---

## 3.1 What do we need?

In [ ]:
import base64
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
import lab

project_client, openai_client = lab.connect()

# Three product photos from our data folder, and the name each one should get.
photos = {
    "alpine-explorer-tent.png": "Alpine Explorer Tent",
    "summit-breeze-jacket.png": "Summit Breeze Jacket",
    "ecofire-camping-stove.png": "EcoFire Camping Stove",
}
print("Photos ready:", list(photos))

## 3.2 Can a model understand a photo?

Our models can take images as input. We send the picture as base64 text inside the request, next to our question.

The photos are real files in `data/images/`. Open one in the editor if you want to see what the model sees (they're large, so we won't paste them into this notebook).

**Predict:** if we ask a small, fast model to name the product in a photo, will it get all three right?

In [ ]:
product_names = sorted(set(photos.values()) | {
    "TrailMaster X4 Tent", "Adventurer Pro Backpack", "TrailLite Daypack",
    "TrekReady Hiking Boots", "TrailWalker Hiking Shoes", "RainGuard Hiking Jacket", "CozyNights Sleeping Bag",
})


def identify(model, filename):
    """Ask a model which product is in a photo. We give it the catalog, so it picks instead of inventing."""
    data = base64.b64encode((lab.DATA / "images" / filename).read_bytes()).decode()
    reply = openai_client.responses.create(
        model=model,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text",
                 "text": "Which product from this catalog is in the photo? Reply with the exact name only.\n"
                         + "\n".join(product_names)},
                {"type": "input_image", "image_url": f"data:image/png;base64,{data}"},
            ],
        }],
    )
    return reply.output_text.strip()


rows = []
for filename, truth in photos.items():
    for model in [lab.SMALL_MODEL, lab.FRONTIER_MODEL]:
        guess = identify(model, filename)
        rows.append({"photo": filename, "model": model, "guess": guess, "correct": guess == truth})

pd.DataFrame(rows)

**Look at the table.** Did the small model get all three? If the small model is good enough at *this* job, we've just found a place to save money, because identifying a product from a photo doesn't need our most expensive model.

## 3.3 What if the answer isn't in the photo?

Now the customer asks: *"Is this one waterproof?"* A picture can't tell us that. The answer lives in the manuals. So we chain two steps:

1. **Look:** a small vision model names the product.
2. **Answer:** our TrailMate agent answers from the manuals, using file search.

Two models, two jobs, one answer.

In [ ]:
filename = "summit-breeze-jacket.png"

# Step 1: the small model looks at the photo.
product = identify(lab.SMALL_MODEL, filename)
print("Step 1, we see:", product)

# Step 2: our agent answers from the manuals, naming the product we found.
reply = lab.ask_agent(openai_client, f"Is the {product} waterproof?")
print("\nStep 2, TrailMate says:\n", reply.output_text)

**Think:**
- Which step used a vision model? Which used our manuals?
- What would have happened if we skipped step 1 and asked the agent "is *this* waterproof?" with only a photo?
- This is v0, with its plain instructions. Does the answer keep the "water-resistant versus waterproof" distinction? We'll come back to that.

## 3.4 How would I send questions to different models?

Different questions deserve different models. Let's build the simplest possible router: a small, cheap model **reads each question and labels it `easy` or `hard`**, and we send hard ones to the big model.

Here is the rule written in plain Python, so we can see every decision.

In [ ]:
def label_difficulty(question):
    """A small model decides if a question is easy (a lookup) or hard (needs reasoning)."""
    reply = openai_client.responses.create(
        model=lab.SMALL_MODEL,
        input=(
            "Label this customer question about outdoor gear. Reply with exactly one word: "
            "easy if it is a single fact lookup, hard if it needs comparing, recommending or reasoning.\n\n"
            + question
        ),
    )
    return reply.output_text.strip().lower()


def pick_model(question):
    return lab.SMALL_MODEL if label_difficulty(question) == "easy" else lab.FRONTIER_MODEL


# Try our rule on six of the practice questions.
practice = lab.load_cases("practice")
sample = [practice[i] for i in [1, 2, 5, 8, 10, 11]]

decisions = pd.DataFrame(
    [{"id": c["id"], "question": c["query"][:60], "we decided": pick_model(c["query"]), "tagged": c["difficulty"]}
     for c in sample]
)
decisions

**Check our work.** Each case in our data carries a difficulty tag we wrote by hand (`tagged`). Does our little router agree with it? Where does it disagree, and who's right?

## 3.5 Is my homemade router good enough?

We built a working router in about ten lines. It's also...

- **Extra work on every request.** We pay for the labeling call and then for the real call.
- **Our own opinion.** "Easy" and "hard" are whatever our prompt makes them.
- **Ours to maintain.** New models arrive, prices change, and we re-tune it forever.
- **Hard to trust.** How do we *know* it routes well? That needs measuring, which is the rest of this lab.

That's exactly the gap a managed **model router** fills. In Module 6 we'll swap our hand-built version for the real one and measure it against the same questions.

## 3.6 So what's the difference, really? (the aha)

Let's put a number on the multi-model half. How much of our traffic did our little router send to the cheap model?

In [ ]:
# Our hand-built router, measured: what share of questions skipped the big model?
share_small = (decisions["we decided"] == lab.SMALL_MODEL).mean()
print(f"Our router sent {round(100 * share_small)}% of these questions to {lab.SMALL_MODEL}.")

> **Aha:** They answer different questions. **Multi-modal** is about *what comes in*: a photo is just another kind of input. **Multi-model** is about *who handles which step*. Our photo example used both, and the router showed where the savings live: a big slice of traffic never needed the big model.

We also took away:

- **Multi-modal** means more than text in. A photo can be the question.
- **Multi-model** means using the right model for each step, not one model for everything.
- A small model is often good enough for the "look" step, which is real savings.
- A hand-built router works, but it's costly to run, subjective and hard to trust without measurement.

That's the stage set. We have models, a way to compare them, and an agent. Time to climb.

**Next:** `04-make-it-work.ipynb`. We'll improve v0 into v1, then ask Foundry to tell us where it's weak.